# Phase 2 - Heart Disease Generative AI Advice

**Course:** SWE485 (Selected Topics in Software Engineering)
**Phase:** 2 (Generative AI)

# The Notebook Overview

### Main Objective
The core objective of this generative AI component is to provide a
personalised health advisory to each patient based on their heart disease
classification result and their individual clinical feature values. Given
a patient's prediction label and clinical measurements, the system
identifies the specific risk factors present in that patient's data and
generates a targeted advisory grounded in those individual values, not
in generic disease labels.

This objective is shared identically across all four prompt templates. The
input is the same for all templates and the output is always a personalised
health advisory. The four templates differ in their prompting techniques and target user profiles, enabling a broader evaluation of how these factors influence the quality, clarity, and depth of the generated advisory.

#### Prompt Inputs

The system uses two primary inputs for each patient assessment: the prediction label and the clinical feature values. Each input plays a specific and deliberate role in enabling the generative model to produce a personalised and clinically grounded advisory.

**Prediction Label**
The prediction label anchors the entire advisory. It tells the system
whether the patient has been classified as having heart disease or not,
which determines the overall direction and tone of the generated advice.
Without this label, the system would have no starting point for framing
the response.

**Clinical Feature Values: Before One Hot Encoding**
The feature values are passed to the generative model in their original
categorical and numerical form, before any one hot encoding is applied.
This is a deliberate and important design decision.

One hot encoding is a preprocessing technique designed for machine
learning algorithms that require numerical inputs. It transforms a
categorical feature such as ChestPainType into multiple binary columns
such as ChestPainType_ASY, ChestPainType_ATA, and so on. While this
transformation is necessary for the supervised model to process the
data, it produces a representation that is fragmented, sparse, and
entirely unreadable to a human or a language model. Traditional
preprocessing methods like one hot encoding can inflate the feature
space and reduce interpretability, particularly with high-dimensional
categorical variables (Ali et al., 2026). Passing one hot encoded
features to a generative AI model would result in the model receiving
inputs like ChestPainType_ASY = 1 and ChestPainType_ATA = 0, which
carry no natural language meaning and cannot be used to generate
coherent clinical explanations.

By contrast, passing the original feature value such as
ChestPainType = ASY allows the generative model to understand the
meaning of the input directly, reason about it using its clinical
knowledge, and incorporate it naturally into the generated advisory.
Large language models are trained on natural language and structured
clinical text, which means they are far better equipped to interpret
and reason about human-readable feature values than binary encoded
representations (Sivarajkumar et al., 2024). This approach ensures
that the advisory is grounded in clinically meaningful inputs rather
than machine-optimised numerical encodings that obscure the original
meaning of the data.

---

> Ali, A.S., et al. (2026). *Unveiling patterns in clinical data:
> exploring the role of large language models and clustering algorithms.*
> Frontiers in Artificial Intelligence, 9, 1737530.
> https://doi.org/10.3389/frai.2026.1737530

> Sivarajkumar, S., et al. (2024). *An empirical evaluation of prompting
> strategies for large language models in zero-shot clinical natural
> language processing.* JMIR Medical Informatics, 12, e55318.
> https://doi.org/10.2196/55318

#### Handling Output Scenarios

The advisory generated by the system is not binary. It does not simply say
"you are healthy" or "you have heart disease." Instead, the response is
always anchored to the patient's specific clinical numbers, and the logic
of the advisory adapts based on two dimensions: the prediction label and
the individual feature values.

**For patients classified as Heart Disease Detected:**
The system identifies which specific clinical features are contributing to
the risk and generates advice directly tied to those features. Two patients
with the same label but different clinical values will always receive
different advice because the advisory is driven by the individual numbers,
not the label alone.

**For patients classified as No Heart Disease Detected:**
The system does not simply reassure the patient. Instead it evaluates the individual feature values using its embedded clinical knowledge to determine whether they are within healthy or concerning ranges. Two distinct
scenarios are handled:

The first scenario is when the patient's feature values are within healthy
ranges. In this case the advisory acknowledges the healthy status and
encourages the patient to maintain their current lifestyle and habits,
reinforcing positive health behaviours.

The second scenario is when one or more feature values are borderline,
meaning they are technically within the healthy classification range but
are approaching concerning thresholds. In this case the system generates
a targeted warning for that specific feature, advising the patient to
monitor it and take preventive action before it progresses. This makes
the advisory genuinely personalised even for patients who are currently
classified as healthy.

This approach reflects the principle that effective health advisory systems
should go beyond binary classification outputs and provide individuals with
actionable, context-aware guidance tailored to their unique clinical profile
(Li et al., 2024). Research in personalised preventive healthcare confirms
that advice anchored to individual clinical values, rather than population-
level generalisations, produces more meaningful and actionable outcomes for
patients (Bajwa et al., 2021). Furthermore, addressing borderline values
in healthy patients aligns with the broader goal of preventive medicine,
where early identification of risk trajectories is considered more valuable
than waiting for a disease threshold to be crossed (Topol, 2019).

---

> Li, Y.H., Li, Y.L., & Wei, M.Y. (2024). *Innovation and challenges of
> artificial intelligence technology in personalized healthcare.*
> Scientific Reports, 14, 18994.
> https://doi.org/10.1038/s41598-024-70073-7

> Bajwa, J., Munir, U., Nori, A., & Williams, B. (2021). *Artificial
> intelligence in healthcare: Transforming the practice of medicine.*
> Future Healthcare Journal, 8(2), e188–e194.
> https://doi.org/10.7861/fhj.2021-0095

> Topol, E.J. (2019). *High-performance medicine: The convergence of
> human and artificial intelligence.* Nature Medicine, 25(1), 44–56.
> https://doi.org/10.1038/s41591-018-0300-7


### Shared Prompt Design Principles
 **System and User Prompt Separation**

The prompt structure adopted across all four templates in this project is divided into two distinct parts: a System Instruction and a User Prompt. This design follows the Google Gemini API documentation, which provides a dedicated `system_instruction` parameter that separates the model's persistent behavioural rules from the variable user message (Google, 2024). The system instruction defines the model's role, tone, language level, and constraints, which are rules that must remain fixed for that template regardless of which patient is being assessed. The user prompt then carries the patient-specific data and the task-related description for each patient assessment.

This separation is not merely a technical convention. It reflects a fundamental principle in prompt engineering: that governance rules and task-specific content should never be mixed in the same layer, as doing so risks inconsistency, ambiguity, and unpredictable model behaviour (Tetrate, 2025). When behavioural rules are embedded within the user prompt, they become prone to being overridden, forgotten, or misinterpreted. By contrast, placing them in a dedicated system instruction ensures they are treated with higher priority by the model and remain stable across all requests.

This principle is especially critical in a medical context. In healthcare AI applications, consistency is not a preference but a safety requirement. A system that gives different advice to similar patients, or that drops its safety constraints when the user prompt is complex, poses a direct risk to patient trust and clinical validity. As noted by Gharib et al. (2024), effective prompt design in healthcare requires that the model's behavioural boundaries are clearly established and reliably maintained across all interactions. Similarly, research on prompt engineering in clinical practice emphasises that small changes in prompt structure can significantly impact output quality and safety in medical settings (Cascella et al., 2024). By anchoring all safety rules, role definitions, and tone constraints within the system instruction, each template in this project ensures that no matter what patient data is passed through the user prompt, the model will always respond within the same controlled and medically appropriate framework.

> Google. (2024). *Gemini API documentation: System instructions.*
> Google AI for Developers.
> https://ai.google.dev/gemini-api/docs/system-instructions

> Tetrate. (2025). *System prompts vs user prompts: Design patterns
> for LLM apps.*
> https://tetrate.io/learn/ai/system-prompts-vs-user-prompts

> Gharib, A.M., et al. (2024). *A guide to prompt design: Foundations
> and applications for healthcare simulationists.* Frontiers in Medicine,
> 11, 1504532.
> https://doi.org/10.3389/fmed.2024.1504532

> Cascella, M., Montomoli, J., Bellini, V. and Bignami, E. (2024).
> *Prompt Engineering in Healthcare.* Electronics, 13(15), 2961.
> https://doi.org/10.3390/electronics13152961


**Role Definition**

The system instruction begins by assigning the model a specific professional role as a clinical decision-support assistant specialized in cardiovascular health analysis. This role is intentionally designed to align with the nature of the dataset used in the system, which includes structured patient information such as key health indicators (e.g., cholesterol levels, blood pressure, and other risk-related features).
Establishing a clear role for the model is considered a best practice in prompt engineering. For example, Google’s prompt design guidelines recommend assigning a role to guide the model’s tone, vocabulary, and depth of explanation (Google, 2024). By framing the model as a healthcare-oriented assistant, the generated responses become more consistent, medically relevant, and aligned with real-world clinical reasoning. This ensures that explanations of predicted outcomes are not only technically accurate but also meaningful to end users.
Furthermore, the assigned role encourages the model to interpret predictions in a way that reflects clinical awareness and responsibility, focusing on explaining possible causes, highlighting key contributing factors, and suggesting reasonable next steps when appropriate. At the same time, the system maintains a balance between professionalism and accessibility, ensuring that explanations remain understandable to non-expert users such as patients or general audiences.
This role-based instruction is particularly important given that the system relies on a predictive model trained on patient data stored in the database . By grounding the model’s responses in this role, the system ensures that all generated explanations are context-aware, structured, and aligned with the intended use of the application as an AI-powered health support tool.
Overall, defining the model’s role enhances the quality, reliability, and clarity of the generated explanations, while also supporting the system’s goal of delivering personalized and user-friendly insights based on patient data.


> Google. (2024). *Introduction to prompting.*
> Google AI for Developers.
> https://ai.google.dev/gemini-api/docs/prompting-intro


### Prompt Template Documentation Structure

Each prompt template in this notebook is documented following a structured, stage-based approach. Rather than simply presenting the prompt and its output, each template is walked through seven progressive stages: Define, Justify, Plan, Build, Test, Evaluate and Reflect. This ensures that every design decision is transparent, deliberate, and well-reasoned. The **Define** stage establishes what the prompting technique is and why it suits the medical domain. The **Justify** stage explains the specific use case and the rationale behind choosing this approach. The **Plan** stage outlines the prompt engineering techniques applied before writing the prompt if applicable. The **Build** stage presents the actual prompt structure. The **Test** stage demonstrates the template with real examples. Finally, the **Evaluate** and **Reflect** stages assess the output quality and acknowledge the template's limitations and ethical considerations. This progressive structure is applied consistently across all four templates to allow meaningful comparison and analysis.

| # | Phase/Stage | Section | What to Write |
|---|-------------|---------|---------------|
| 1 | | **ID** | A number |
| 2 | Define | **Approach Definition** | Define the prompting technique in simple terms. What is it? How does it work? |
| 3 | Define | **Relation to Medical Field** | Why does this specific technique make sense in a healthcare context? Connect the technique's characteristics to medical needs. |
| 4 | Justify | **Intended Use Case** |  Who is the user? What do they input? What do they need as output? |
| 5 | Justify | **Design Rationale** | Why did we choose this approach over other techniques? What advantages does it have for your specific problem? |
| 6 | Plan | **Prompt Engineering Techniques Applied** | Explain any additional good practices deliberately applied in the prompt. Why was each technique included? |
| 7 | Build | **Prompt Structure** | The actual prompt we send to the AI model with placeholders in `{}` for parts that change per user. |
| 8 | Test | **Input / Output Example** | Fill the placeholders with real examples and show the actual AI response you received (3 test cases). |
| 9 | Evaluate | **Evaluation Summary** | Rate the response across the decided criteria. |
| 10 | Reflect | **Assumptions & Limitations** | What must be true for this template to work? Where does it fail or produce weak results? It also includes any ethical concerns. |

# Generative AI Model Setup

### Load the Best Model from Phase 1

In [2]:
import pickle

# Load the best model saved from Phase 1 (XGBoost model)
with open("Supervised_Learning/models/best_xgb.pkl", "rb") as f:
    model = pickle.load(f)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.discriminant_analysis import StandardScaler
from sklearn.preprocessing import RobustScaler

#Method to get prediction from the model, this will be used in the prompt for Gemini

#Step 1:
# Fit scalers on the same training data statistics
# Age → StandardScaler | RestingBP → RobustScaler | MaxHR → StandardScaler | Oldpeak → RobustScaler
raw_df =  pd.read_csv("Dataset/preprocessed_heart_data.csv")

#WE NEED TO CHECK THIS!!
age_scaler      = StandardScaler().fit(raw_df[['Age']])
resting_bp_scaler = RobustScaler().fit(raw_df[['RestingBP']])
max_hr_scaler   = StandardScaler().fit(raw_df[['MaxHR']])
oldpeak_scaler  = RobustScaler().fit(raw_df[['Oldpeak']])

def get_prediction(inputs):

    #Step 2: Cholesterol → category then drop raw value 
    chol = inputs["Cholesterol"]
    chol_bins   = [0, 200, 240, np.inf]
    chol_labels = ["Desirable", "Borderline High", "High"]
    chol_cat    = pd.cut([chol], bins=chol_bins, labels=chol_labels, right=False)[0]

    # Step 3:Build encoded row 
    encoded = {
        "Age":                              inputs["Age"],
        "Sex":                              1 if inputs["Sex"] == "Male" else 0,
        "RestingBP":                        inputs["RestingBP"],
        "FastingBS":                        1 if inputs["FastingBS"] > 120 else 0,
        "MaxHR":                            inputs["MaxHR"],
        "ExerciseAngina":                   1 if inputs["ExerciseAngina"] == "Yes" else 0,
        "Oldpeak":                          inputs["Oldpeak"],
        # ChestPainType one-hot
        "ChestPainType_ASY":                1 if inputs["ChestPainType"] == "ASY" else 0,
        "ChestPainType_ATA":                1 if inputs["ChestPainType"] == "ATA" else 0,
        "ChestPainType_NAP":                1 if inputs["ChestPainType"] == "NAP" else 0,
        "ChestPainType_TA":                 1 if inputs["ChestPainType"] == "TA"  else 0,

        # RestingECG one-hot
        "RestingECG_LVH":                   1 if inputs["RestingECG"] == "LVH"    else 0,
        "RestingECG_Normal":                1 if inputs["RestingECG"] == "Normal" else 0,
        "RestingECG_ST":                    1 if inputs["RestingECG"] == "ST"     else 0,

        # ST_Slope one-hot
        "ST_Slope_Down":                    1 if inputs["STSlope"] == "Down" else 0,
        "ST_Slope_Flat":                    1 if inputs["STSlope"] == "Flat" else 0,
        "ST_Slope_Up":                      1 if inputs["STSlope"] == "Up"   else 0,

        # Chol_category one-hot
        "Chol_category_Desirable":          1 if chol_cat == "Desirable"       else 0,
        "Chol_category_Borderline High":    1 if chol_cat == "Borderline High" else 0,
        "Chol_category_High":               1 if chol_cat == "High"            else 0,
    }

    df = pd.DataFrame([encoded])

    # Step 4: Apply same scalers from Phase 1 
    df["Age"]       = age_scaler.transform(df[["Age"]])
    df["RestingBP"] = resting_bp_scaler.transform(df[["RestingBP"]])
    df["MaxHR"]     = max_hr_scaler.transform(df[["MaxHR"]])
    df["Oldpeak"]   = oldpeak_scaler.transform(df[["Oldpeak"]])

    result = model.predict(df)
    return "Heart Disease" if result[0] == 1 else "No Heart Disease"

In [ ]:
#JUST AN EXAMPLE, EACH ONE OF US WILL DO LIKE THIS WHEN HER PROMPT IS READY, SHE WILL PREPARE THE INPUT, THEN SEND TO OUR MODEL TO GET THE 
#PREDICTION THAT WILL BE SEND WITHIN THE PROMPT
inputs = {
    "Age":              55,
    "Sex":              "M",
    "ChestPainType":  "ATA",
    "RestingBP":       140,
    "Cholesterol":      240,
    "FastingBS":       1,
    "RestingECG":      "Normal",
    "MaxHR":           150,
    "ExerciseAngina":  "Y",
    "Oldpeak":          2.3,
    "STSlope":         "Flat"
}

inputs["prediction"] = get_prediction(inputs)
print(inputs["prediction"])

### Generative AI setup

In [ ]:
#Imports & Client Setup
from google import genai
from dotenv import load_dotenv

load_dotenv()
client = genai.Client()
MODEL = "gemini-2.5-flash"

None


In [24]:
#Shared Method 
def call_gemini(system_prompt, user_prompt):
    response = client.models.generate_content(
        model=MODEL,
        contents=user_prompt,
        config={"system_instruction": system_prompt}
    )
    return response.text

In [ ]:
#JUST AN EXAMPLE, EACH ONE OF US WILL DO LIKE THIS WHEN HER PROMPT IS READY, SHE WILL PREPARE THE INPUT, THE USER PROMPT, THE SYSTEM PROMPT
#THEN CALL THIS METHOD THAT IS ALREADY PREPARED FOR YOU GUYS
inputs = {
    "prediction":       "No Heart Disease",
    "Age":              55,
    "Sex":              "Male",
    "ChestPainType":  "ATA",
    "RestingBP":       140,
    "Cholesterol":      240,
    "FastingBS":       1,
    "RestingECG":      "Normal",
    "MaxHR":           150,
    "ExerciseAngina":  "Yes",
    "Oldpeak":          2.3,
    "STSlope":         "Flat"
}

SYSTEM_T1 = """You are a medical assistant that helps explain heart disease risk to patients."""

USER_T1 = """A patient has been assessed and the result is: {prediction}.
Patient details: Age={Age}, Sex={Sex}, Chest Pain={ChestPainType}, Resting BP={RestingBP},
Cholesterol={Cholesterol}, Fasting BS={FastingBS}, Resting ECG={RestingECG},
Max HR={MaxHR}, Exercise Angina={ExerciseAngina}, Oldpeak={Oldpeak}, ST Slope={STSlope}.
Please provide a brief and simple explanation of this result."""

result_t1 = call_gemini(SYSTEM_T1, USER_T1.format(**inputs))
print(result_t1)

# Generated Knowledge Prompting

 ### Template ID 
 1

---
### Approach Definition

Generated Knowledge Prompting is a prompt engineering technique introduced
by Liu et al. (2022) that instructs the model to first generate relevant
knowledge about the topic before producing its final response. Rather than
asking the model to answer directly, the prompt is structured in two
internal stages within a single request: the model first surfaces and
organises what it knows about the subject, and then uses that
self-generated knowledge as the foundation for its final output.

The core idea behind this technique is that large language models contain
vast amounts of knowledge from their training, but do not always surface
the most relevant information when asked to respond directly. By explicitly
instructing the model to generate knowledge first, the prompt forces a
broader and more deliberate retrieval of relevant information before any
conclusion is produced. This generated knowledge then sits within the
model's context during the response generation phase, ensuring that the final
response is grounded in a richer and more organised understanding of the
topic (Liu et al., 2022). This approach does not require access to an
external knowledge base or any task-specific training, making it flexible
and practical for a wide range of applications.

> Liu, J., Liu, A., Lu, X., Welleck, S., West, P., Le Bras, R., Choi, Y.,
> and Hajishirzi, H. (2022). *Generated Knowledge Prompting for Commonsense
> Reasoning.* Proceedings of the 60th Annual Meeting of the Association for
> Computational Linguistics, 3154–3169.
> https://doi.org/10.18653/v1/2022.acl-long.225


---

### Relation to Medical Field

In a medical context, generating knowledge before producing a response is
particularly valuable because clinical reasoning is rarely a direct
input-to-output process. A clinician examining a patient's cardiovascular
data does not immediately jump to a conclusion. They first consider what
each measurement means, how it relates to known clinical patterns, and what
risks it might indicate, before forming an assessment and recommending
next steps. Generated Knowledge Prompting mirrors this interpretive process
by instructing the model to first surface and organise its clinical
knowledge about the patient's specific feature values before producing
the final advisory.

This is especially important in cardiovascular health, where the clinical
significance of a measurement depends heavily on context. A resting blood
pressure of 145 mmHg, for example, cannot be interpreted in isolation. The
model needs to draw on its knowledge of what that value means, how it
relates to cardiovascular risk, and how it interacts with other features
in the patient's profile before it can generate advice that is genuinely
meaningful. By explicitly instructing the model to generate this contextual
knowledge first, the prompt ensures that the final advisory is grounded in
a deliberate clinical reasoning step rather than a surface level response.


---


### Intended Use Case

 **1- Deployment Environment**

The system is planned to be deployed as an advisory tool within a hospital
mobile application. Patients access it from their personal smartphones
after obtaining their clinical data, either from a recent cardiovascular
test or by completing one at the hospital. The patient enters their
clinical values into the tool and receives a personalized advisory
directly on the same screen.

 **2- The User**

The primary user of this template is a **first-time patient** who is
encountering the system and their clinical results for the very first
time. This patient arrives with no frame of reference for what their
numbers mean. A resting blood pressure of 145 mmHg is just a number to
them. An Oldpeak value of 2.3 carries no weight without context. A
cholesterol category label means nothing without knowing what it implies
for their health. Providing advice to this patient without first
establishing what their values mean would produce a response they cannot
connect to their own situation, and advice that cannot be connected to
is advice that will not be acted on. Research in health education
confirms that patients who understand the meaning of their clinical
measurements are significantly more likely to follow through on health
recommendations than those who receive advice without explanation
(Berkman et al., 2011).

< Berkman, N.D., et al. (2011). *Health literacy interventions and
> outcomes: An updated systematic review.* Evidence Report/Technology
> Assessment, 199, 1–941.

***Medical background:*** No medical background is assumed. The gap
between what the patient's values say and what they mean to that patient
is treated as the central design constraint of this template. Before any
recommendation can land meaningfully, the patient needs enough
foundational understanding to connect the advice back to their own
situation rather than receiving it as an instruction they must simply
trust.

***Emotional state:*** First-time users often arrive uncertain and quietly
apprehensive. They may not know what they are looking for or what a
concerning result would even look like. The tone of the advisory must
therefore be steady and educational rather than clinical or alarming.
The goal is not to reassure unconditionally or to alarm unnecessarily,
but to leave the patient feeling genuinely informed and oriented enough
to take the next right step.

**3- Inputs**

3.1- The patient heart disease classification result 

3.2- The patient clinical feature values


**4- Output**

A single advisory paragraph written in plain, accessible language.
Although the model generates clinical knowledge internally as part of
its knowledge generation process, none of this internal knowledge generation is
visible to the patient. The output reads as a clean, direct advisory,
but because it was built on top of a deliberate knowledge generation
step, it is more grounded in the patient's specific values and more
educationally coherent than a direct response would be for someone
encountering their clinical data for the first time. The patient leaves
the interaction not only with advice to follow but with enough implicit
understanding of their values to begin asking the right questions at
their next clinical appointment.


---

### Design Rationale

**Advantage 1 — Grounded knowledge generation before response:**
Generated Knowledge Prompting instructs the model to first surface and
organise relevant knowledge about the input before producing its final
response, ensuring the output is grounded in an explicit knowledge generation step
rather than a direct leap from input to answer (Liu et al., 2022). In a
heart disease advisory system, this matters because the clinical
significance of a patient's feature values cannot be determined without
first interpreting what those values mean in a cardiovascular context.
A model that jumps directly to advice without first generating knowledge about the
patient's specific measurements risks producing generic or misaligned
recommendations that do not reflect the patient's actual profile.

**Advantage 2 — Improved contextual relevance for individual patients:**
By generating knowledge that is specific to the input provided, the
technique ensures that the final response is tailored to the individual
patient rather than relying on broad generalisations (Liu et al., 2022).
In a heart disease advisory system where two patients can share the same
prediction label but have entirely different clinical profiles, this
matters directly. A patient with a borderline resting blood pressure and
a flat ST slope requires fundamentally different advice from a patient
with a normal blood pressure and an asymptomatic chest pain type, even
if both are classified as No Heart Disease Detected. Generated Knowledge Prompting ensures the model generates contextual
knowledge about the specific values present before writing the advisory,
producing a response that reflects the individual rather than the label.

**Advantage 3 — No dependency on predefined examples:**
Generated Knowledge Prompting does not require carefully crafted
input-output examples to guide the model's behaviour. The model draws
on its own clinical knowledge to reason about the patient's data, making
the technique flexible and robust across the full range of possible
patient profiles (Liu et al., 2022). In a heart disease advisory system
where patient feature values vary continuously across a wide range,
providing representative examples that cover all meaningful combinations
is not practical. Generated Knowledge Prompting eliminates this
dependency by allowing the model to generate the contextual knowledge
it needs dynamically from the input itself.


> Liu, J., Liu, A., Lu, X., Welleck, S., West, P., Le Bras, R., Choi, Y.,
> and Hajishirzi, H. (2022). *Generated Knowledge Prompting for Commonsense
> Reasoning.* Proceedings of the 60th Annual Meeting of the Association for
> Computational Linguistics, 3154–3169.
> https://doi.org/10.18653/v1/2022.acl-long.225

In [ ]:
inputs = {
    "Age":              55,
    "Sex":              "Male",
    "ChestPainType":    "ATA",
    "RestingBP":        140,
    "Cholesterol":      240,
    "FastingBS":        1,
    "RestingECG":       "Normal",
    "MaxHR":            150,
    "ExerciseAngina":   "Yes",
    "Oldpeak":          2.3,
    "STSlope":          "Flat"
}

# Get prediction from the model and add it to inputs
inputs["prediction"] = get_prediction(inputs)




SYSTEM_T4 = """You are a clinical decision-support assistant specialised in cardiovascular health. 
You communicate in plain, accessible language suitable for patients with no medical background.

You follow a two-stage internal process for every patient assessment:

STAGE 1 — GENERATE CLINICAL KNOWLEDGE (INTERNAL ONLY — DO NOT SHOW TO PATIENT):
Before writing any advice, you must first reason internally about the patient's 
clinical values. For each feature provided, generate your clinical knowledge about 
what that value means, whether it falls within a healthy, borderline, or concerning 
range, and what its implications are for cardiovascular health. This stage must 
never appear in your response. It is a silent reasoning step only.

STAGE 2 — WRITE THE ADVISORY (THIS IS THE ONLY THING YOU OUTPUT):
Using the knowledge you generated in Stage 1, write a single personalised advisory 
paragraph for the patient. The advisory must:
- Be grounded in the patient's specific values, not in generic disease labels
- Reflect the prediction result and the individual feature values
- If the result is Heart Disease Detected: explain which specific values are 
  contributing to the risk and what the patient should do next
- If the result is No Heart Disease Detected and all values are healthy: 
  reassure the patient and encourage them to maintain their current lifestyle
- If the result is No Heart Disease Detected but one or more values are borderline: 
  acknowledge the healthy result but flag the specific borderline values and 
  provide targeted preventive advice for each one
- Use a calm, grounding, and educational tone
- Never use clinical jargon without explanation
- Never provide a diagnosis
- Always recommend consulting a medical professional for further guidance

YOUR RESPONSE MUST CONTAIN ONLY THE ADVISORY PARAGRAPH. 
NOTHING ELSE. NO LABELS. NO HEADERS. NO STAGE TITLES."""


USER_T4 = """Patient Assessment Result: {prediction}

Patient Clinical Values:
- Age: {Age} years
- Sex: {Sex}
- Chest Pain Type: {ChestPainType}
- Resting Blood Pressure: {RestingBP} mmHg
- Cholesterol: {Cholesterol} mg/dL
- Fasting Blood Sugar above 120 mg/dL: {FastingBS} (0 = No, 1 = Yes)
- Resting ECG: {RestingECG}
- Maximum Heart Rate: {MaxHR} bpm
- Exercise-Induced Angina: {ExerciseAngina}
- Oldpeak (ST Depression): {Oldpeak}
- ST Slope: {STSlope}

Based on this patient's specific values, generate your internal clinical knowledge 
about each feature first, then write a single personalised health advisory paragraph 
for this patient in plain language."""


result_t4 = call_gemini(SYSTEM_T4, USER_T4.format(**inputs))
print(result_t4)

Your recent health assessment indicates that there are signs suggesting heart disease, and this is primarily linked to several factors in your profile. You’ve experienced chest discomfort, both spontaneously and during exercise, which strongly suggests your heart might not be getting enough blood flow when it needs it. Additionally, your resting blood pressure of 140 mmHg is high, and your cholesterol level of 240 mg/dL is elevated. Your fasting blood sugar also shows elevated levels, which can contribute to heart health challenges over time. Furthermore, the changes observed on your heart tracing (ECG) during exercise, specifically the ST depression and flat ST slope, are important indicators that your heart muscle may not be receiving adequate oxygen during physical activity. While your resting ECG was normal, these other significant findings collectively point towards a need for further evaluation. It is very important that you consult with your doctor as soon as possible to discuss

# Chain-of-Thought Prompting

### Template ID 
2

---

### Approach Definition

Chain-of-thought (CoT) prompting is a prompt engineering technique in which a language model is instructed to produce a series of intermediate reasoning steps before arriving at a final answer. Rather than mapping an input directly to an output in a single pass, the model externalizes its thinking process, each step builds on the previous one, and the final answer is explicitly grounded in the visible reasoning chain. IBM describes CoT as a technique that simulates human-like reasoning by breaking down elaborate problems into manageable, intermediate steps that sequentially lead to a conclusive answer (IBM, 2024). The technique is activated by including a structured reasoning directive in the prompt, most commonly the phrase "think step by step," which signals to the model that intermediate steps are required as part of the response rather than just a final conclusion.


> IBM. (2024). *What is chain of thought (CoT) prompting?* IBM Think.  
> __https://www.ibm.com/think/topics/chain-of-thoughts__

---

### Relation to Medical Field

The structure of chain-of-thought prompting mirrors the way clinicians think when making a diagnosis. A doctor assessing a patient for cardiovascular disease does not jump straight from test results to a treatment decision. Patel and Groen (1986) studied how cardiologists reason through complex cases and found that accurate diagnoses were reached by moving in one direction only: from clinical findings, step by step, toward a conclusion. Each step followed from the one before it, building on the available evidence rather than starting from an assumed answer. This is exactly how chain-of-thought prompting works: the model moves forward through a series of reasoning steps, each grounded in the previous one, without going back to revise earlier conclusions, making CoT a naturally suitable technique for clinical decision-making tasks.

> Patel, V. L., & Groen, G. J. (1986). Knowledge based solution strategies in medical reasoning. *Cognitive Science*, *10*(1), 91–116. 
> https://doi.org/10.1207/s15516709cog1001_4
---

### Intended Use Case

**1- Deployment Environment**

The system is planned to be deployed as an advisory tool within a hospital
mobile application. Patients access it from their personal smartphones
after obtaining their clinical data, either from a recent cardiovascular
test or by completing one at the hospital. The patient enters their
clinical values into the tool and receives a personalized advisory
directly on the same screen. 

**2- The User**

The primary user of this template is a detail-oriented patient who approaches their health result with curiosity and a desire to understand rather than simply act. This patient is likely to be someone who has prior experience with cardiovascular health (either through a personal history of monitoring their health closely, a family member with heart disease, or a general habit of researching their medical information). They are comfortable reading a longer, structured response and will invest the time to follow a step-by-step explanation from beginning to conclusion. 
Research in patient health literacy confirms that a significant proportion of patients actively prefer detailed explanations of their results, as understanding the reasoning behind a recommendation increases their confidence in acting on it and their trust in the system delivering it (Zolnierek and Dimatteo, 2009).
> Zolnierek, K.B.H. and DiMatteo, M.R. (2009). *Physician communication
> and patient adherence to treatment: A meta-analysis.* Medical Care,
> 47(8), 826–834.
> https://doi.org/10.1097/MLR.0b013e31819a5acc

***Medical background:*** While no formal medical background is assumed, this patient is health-literate. They are capable of understanding concepts such as blood pressure ranges, cholesterol categories, and the general relationship between lifestyle factors and cardiovascular risk. The advisory may therefore use slightly more specific language than the zero-shot template, as long as each term is explained in context. (Is it okay to compare here?)

***Emotional state:*** This patient is engaged rather than acutely anxious. They are seeking information and understanding, which means a longer, more structured response will not increase their distress. However, the tone must still remain calm and professional. The visibility of the reasoning steps must never make the output feel like a clinical verdict.

**3- Inputs**

3.1 The patient heart disease classification result 

3.2 The patient clinical feature values


**4- Output**

A structured response consisting of a brief per-feature assessment followed by a synthesised final advisory paragraph. The reasoning steps walk through the patient's most significant clinical values one by one, identifying what each means and how it contributes to the overall picture. The final paragraph then synthesises these steps into a coherent, actionable recommendation. The visible reasoning is written in plain language throughout so that the patient can follow each step without a medical background. Scrolling is acceptable here because the patient prefers this level of details. 

---

### Design Rationale

**Advantage 1 — Improved accuracy on complex reasoning tasks:**
Chain-of-thought improves model performance on complex tasks by breaking
them down into smaller, manageable steps, with each intermediate step
acting as a checkpoint where errors can be caught before they carry
forward into the final output (DataCamp, 2024). In a heart disease
advisory system, this matters because an error made early (such as
misreading a borderline cholesterol value or misclassifying a resting
ECG result) will corrupt every reasoning step that follows it. By
making each step visible and sequential, chain-of-thought allows such
errors to be identified at the point they occur rather than arriving
silently embedded in a final recommendation.

**Advantage 2 — Transparency and explainability:** By generating visible
intermediate reasoning steps, chain-of-thought makes the decision-making
process auditable and easier to follow for non-expert users (IBM, 2024).
In a heart disease advisory system where the intended users have no
medical background, this is a functional requirement. A patient who
receives only a label or a conclusion cannot understand what it
means for them personally, identify which of their values drove the
outcome, or communicate the basis of the advisory to a doctor. By
walking through each feature in plain, step-by-step reasoning before
reaching a conclusion, chain-of-thought produces an output that a
non-specialist can follow, question, and act on, rather than simply
accept or dismiss.

**Advantage 3 — Multistep reasoning capability:** Chain-of-thought
handles tasks that require multistep reasoning by systematically working
through each component of a problem before reaching a conclusion, leading
to more reliable outputs (IBM, 2024). In cardiovascular test,
reliability is critical where a premature conclusion or a skipped step can
result in an advisory that does not reflect the patient's actual
condition. By structuring the output as a sequence of dependent reasoning
steps, chain-of-thought ensures the model cannot arrive at a conclusion
before all components of the problem have been addressed, making the
final output consistent and reproducible regardless of how the input
values are distributed.

**Advantage 4 — Attention to detail:** The step-by-step nature of
chain-of-thought encourages detailed breakdowns of complex inputs,
ensuring each component is examined and accounted for before a conclusion
is reached (IBM, 2024). In cardiovascular test, where no
single factor is decisive on its own, this level of detail is essential.
The ACC/AHA Pooled Cohort Equations calculate cardiovascular risk from
multiple variables whose combined effect is non-additive, meaning the
contribution of any one feature depends on the values of the others
(Goff et al., 2014). Chain-of-thought ensures that no feature is skipped
or treated as insignificant, making it the most detail-preserving
technique available for this task.


> DataCamp. (2024). *Chain-of-thought prompting*. 
> https://www.datacamp.com/tutorial/chain-of-thought-prompting

> IBM. (2024). *Chain of thoughts*. IBM Think. 
> https://www.ibm.com/think/topics/chain-of-thoughts

> Goff, D. C., Lloyd-Jones, D. M., Bennett, G., Coady, S., D'Agostino, R. B.,  Gibbons, R., Greenland, P., Lackland, D. T., Levy, D., O'Donnell, C. J.,  Robinson, J. G., Schwartz, J. S., Shero, S. T., Smith, S. C., Sorlie, P., Stone, N. J., & Wilson, P. W. F. (2014). 2013 ACC/AHA guideline on the assessment of cardiovascular risk: A report of the American College of Cardiology/American Heart Association Task Force on Practice Guidelines. *Journal of the American College of Cardiology*, *63*(25), 2935–2959. 
> https://doi.org/10.1016/j.jacc.2013.11.005

---


# Few-Shot Prompting

### Template ID
3

---

### Approach Definition

Few-shot prompting is particularly well suited to the medical domain because clinical reasoning rarely happens in isolation. When a clinician assesses a new cardiovascular patient, they do not interpret the case from scratch. Instead, they draw on previously encountered patients with similar profiles, comparing the current case against familiar patterns before reaching a conclusion. A cardiologist who has seen many patients with high blood pressure, elevated cholesterol, and chest pain will immediately recognize that combination as a risk pattern worth taking seriously, not because of a single number, but because of how the full picture matches cases seen before(Elstein, 2002).

This is exactly how few-shot prompting works(Brown et al., 2020). By placing a small number of labeled patient records inside the prompt, the model is shown how similar cardiovascular profiles have been interpreted in the past. Each example acts as a reference point, guiding the model to recognize patterns across multiple variables rather than reacting to any single measurement in isolation. In that sense, few-shot prompting functions as a lightweight case-based reasoning mechanism, one that mirrors the comparative, pattern-driven logic that underlies real clinical judgment in cardiovascular care.

> Brown, T. et al. (2020). *Language Models are Few-Shot Learners*. NeurIPS.  
> https://arxiv.org/abs/2005.14165

> Elstein, A. S. (2002). *Clinical problem solving and diagnostic decision making*.  
> https://pmc.ncbi.nlm.nih.gov/articles/PMC1122649/
---

### Relation to the Medical Field

Few-shot prompting is well suited to the medical domain because medical decision-making often depends on recognizing patterns across multiple patient indicators rather than relying on a single feature. In a heart disease dataset, prediction is not based on only age, cholesterol, or blood pressure alone, but on the way these variables interact with each other.

This makes few-shot prompting relevant for a heart disease advisory system. By showing the model a few examples of patient records together with their corresponding outcomes, the model can learn the expected reasoning pattern for similar cardiovascular cases. In other words, the prompt itself becomes a lightweight guide that teaches the model how to interpret combinations of medical features in a structured and task-specific way.

---

### Intended Use Case

**1- Deployment Environment**

The system is designed to be deployed as a heart disease advisory tool within a healthcare mobile application. Patients can access it through their smartphones after obtaining their clinical data, either from a hospital test or by manually entering their values into the system. The advisory is generated instantly and displayed on the same screen.



**2- The User**

The primary user of this template is a returning patient who has used the healthcare application before and has already received at least one prior advisory through the system. This user is familiar with the general style, tone, and structure of the generated response, and they return expecting a consistent and recognisable experience.
Unlike a first-time user, this patient does not need the system to introduce the format from the beginning. Instead, they benefit from receiving advice in a stable and predictable structure that allows them to compare their current result with previous advisories, notice changes in their condition, and follow their health status over time.

***Medical background:***  

No medical expertise is assumed. However, because the user has prior experience with the system, the response does not need to introduce or justify its structure, and can instead focus directly on the advisory content. The output should avoid complex terminology and focus on simple explanations supported by examples.

***Emotional state:***  
The patient may still feel some concern when reviewing a health-related result, but they are likely to approach the output with greater familiarity than a new user. For that reason, the response should remain supportive and clear while focusing on reassurance through consistency and readability.



**3- Inputs**

3.1 The patient heart disease classification result  

3.2 The patient clinical feature values  




**4- Output**

A structured but concise response that follows the pattern demonstrated in the examples. The output includes:
- A short interpretation of the patient’s condition  
- A brief explanation based on the most relevant features  
- A clear and simple advisory  

The response should remain consistent with the format shown in the examples, ensuring readability and ease of understanding.

---

### Design Rationale
**Advantage 1 —  Better adaptation to the task:**
Few-shot prompting helps the model adapt to the specific prediction task by showing it exactly how inputs and outputs should look. This is important in the heart disease dataset because the model needs to deal with structured patient attributes such as age, sex, chest pain type, resting blood pressure, cholesterol, fasting blood sugar, and other clinical indicators. A few examples help the model understand how these features are mapped to the target class.(OpenAI, 2023)

**Advantage 2 — Improved consistency of outputs:** 
When examples are included in the prompt, the model is more likely to respond in a consistent format. This is useful in a medical application because predictions should be presented clearly and uniformly. For example, if each example follows the same structure of patient data, prediction, and short explanation, the model is more likely to generate outputs that are easier to compare, evaluate, and interpret.

**Advantage 3 — Useful when labeled examples are meaningful:** 
Our dataset already contains labeled cases, which makes few-shot prompting a natural choice. Since the target variable indicates the presence or absence of heart disease, example records can be selected directly from the dataset to guide the model. This allows the prompting technique to align closely with the supervised learning setting, where the model benefits from seeing representative examples before handling unseen cases.

**Advantage 4 — Supports pattern recognition in tabular medical data:**
The heart disease dataset is structured and feature-based, which makes it suitable for few-shot prompting. The model can observe how certain combinations of values are associated with different outcomes and then apply that learned pattern to a new case. This is especially helpful in healthcare tasks, where risk is often determined by the combined effect of multiple variables rather than one isolated measurement.(Goff et al., 2014)


> OpenAI. (2023). *Prompt Engineering Guide*.  
> https://platform.openai.com/docs/guides/prompt-engineering

> Goff, D. C. et al. (2014). *2013 ACC/AHA Guideline on the Assessment of Cardiovascular Risk*. Journal of the American College of Cardiology.  
> https://doi.org/10.1016/j.jacc.2013.11.005

